# EP.1 Mixture of Experts：路由、容量与共享专家

配套文章：[MoE 技术原理](https://momoyeyu.github.io/posts/llm-moe/)。上一章：[00_transformer.ipynb](00_transformer.ipynb)；下一章：[02_normalization.ipynb](02_normalization.ipynb)。本章复用上一章首次展开的 `FFN`，以及其中定义的 `set_seed` / `show` 实验工具，把 token-wise FFN 扩展成稀疏专家池。核心问题不是“多放几个 MLP”，而是：每个 token 选谁、如何平衡、溢出如何处理。

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "slm").is_dir():
    root = next((parent for parent in root.parents if (parent / "slm").is_dir()), None)
if root is None:
    raise RuntimeError("请从仓库根目录或 notebooks 目录运行")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from slm import set_seed, show

torch.set_num_threads(1)
set_seed(0)
show("Python", sys.version.split()[0])
show("PyTorch", torch.__version__)
show("device", "cpu")
from slm import FFN

Python                       3.12.12
PyTorch                      2.14.1
device                       cpu


## 1. Dense 与专家 FFN 的矩阵乘成本

先只数两个线性层的乘加：$2D_{model}D_{ff}$。

In [2]:
def ffn_macs(d_model: int, d_ff: int) -> int:
    return 2 * d_model * d_ff

In [3]:
# GPT-2 small：d_model=768, d_ff=3072
d_model, d_ff, N, K = 768, 3072, 8, 2
dense = ffn_macs(d_model, d_ff)
show("Dense FFN MACs", f"2·{d_model}·{d_ff} = {dense:,}")

# 等参拆分：每 Expert 缩小为 d_ff/N，N 个合计与 Dense 一致，激活 K 个 = K/N
per_expert = ffn_macs(d_model, d_ff // N)
show("equal split per-expert", f"{d_model}->{d_ff // N}->{d_model} = {per_expert:,}")
show("  total N experts", f"{N * per_expert:,} = Dense")
show("  active K experts", f"{K * per_expert:,} = {K}/{N} × Dense")

# 扩容拆分：每 Expert 保持 d_ff，激活 K 个就是 K 倍 Dense
show("expand split per-expert", f"{ffn_macs(d_model, d_ff):,} = Dense")
show("  active K experts", f"{K * ffn_macs(d_model, d_ff):,} = {K} × Dense")

Dense FFN MACs               2·768·3072 = 4,718,592
equal split per-expert       768->384->768 = 589,824
  total N experts            4,718,592 = Dense
  active K experts           1,179,648 = 2/8 × Dense
expand split per-expert      4,718,592 = Dense
  active K experts           9,437,184 = 2 × Dense


这些数只覆盖专家内部 dense matmul MAC，不含 router、bias、dispatch、通信或 padding。所谓“等参拆分”也不是整个模型参数量严格相等。

## 2. Router：概率、Top-K 索引与门权重

Router 先产生所有专家概率 `[T,N]`，再取每个 token 的 Top-K 索引 `[T,K]`，并在选中项内重新归一化权重。

In [4]:
class Router(nn.Module):
    def __init__(self, d_model: int, num_experts: int, top_k: int) -> None:
        super().__init__()
        self.top_k = top_k
        self.weight = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits = self.weight(x)                                              # [T, N]
        probs = torch.softmax(logits, dim=-1)                                # [T, N]
        topk_probs, topk_idx = probs.topk(self.top_k, dim=-1)                # [T, K], [T, K]
        routing_weights = topk_probs / topk_probs.sum(dim=-1, keepdim=True)  # [T, K]
        return probs, topk_idx, routing_weights

In [5]:
import torch


set_seed(0)
router = Router(d_model=16, num_experts=8, top_k=2)
probs, topk_idx, weights = router(torch.randn(4, 16))

show("probs[0]", probs[0].tolist())
show("Top-2 idx[0]", topk_idx[0].tolist())
show("weights[0]", weights[0].tolist())
show("weights row sums", weights.sum(-1).tolist())

probs[0]                     [0.10797390341758728, 0.10780945420265198, 0.19335488975048065, 0.22987209260463715, 0.06732828170061111, 0.062338899821043015, 0.14808636903762817, 0.08323615044355392]
Top-2 idx[0]                 [3, 2]
weights[0]                   [0.5431413650512695, 0.4568586051464081]
weights row sums             [1.0, 1.0, 1.0, 1.0]


每行概率和与选中权重和都为 1。特别地 K=1 时，重归一化后的主分支门权重恒为 1，主损失不能通过这个标量训练选择；辅助负载损失因而很重要。

## 3. 负载均衡损失

$f_i$ 是被 Top-K 选中专家 $i$ 的 token 比例，$P_i$ 是其平均路由概率：

$$L_{aux}=\alpha N\sum_i f_iP_i.$$

每个 token 选择 K 个专家，所以 $\sum_i f_i=K$；均衡时损失是 $\alpha K$，不是 $\alpha$。

In [6]:
def load_balancing_loss(
    probs: torch.Tensor,    # [T, N]
    topk_idx: torch.Tensor, # [T, K]
    alpha: float,
) -> torch.Tensor:
    N = probs.size(1)
    dispatch_mask = torch.zeros_like(probs)    # [T, N]
    dispatch_mask.scatter_(1, topk_idx, 1.0)   # [T, N]
    f = dispatch_mask.mean(dim=0)              # [N]
    P = probs.mean(dim=0)                      # [N]
    return alpha * N * (f * P).sum()

In [7]:
import torch


set_seed(0)
T, N, K, alpha = 64, 8, 2, 0.01

# 均衡：每个 Expert 的 f_i = K/N，损失恰好为 αK
probs = torch.full((T, N), 1 / N)
topk_idx = torch.arange(T * K).reshape(T, K) % N
show("balanced L_aux", f"{load_balancing_loss(probs, topk_idx, alpha).item():.4f} = αK = {alpha * K}")

# 偏斜：所有 token 都挤到前两个 Expert，f_0 = f_1 = 1
probs = torch.rand(T, N).softmax(-1).requires_grad_(True)
topk_idx = torch.zeros(T, K, dtype=torch.long)
topk_idx[:, 1] = 1
loss = load_balancing_loss(probs, topk_idx, alpha)
loss.backward()

# f 不可微只作系数；∂L/∂p_{t,i} = αN·f_i/T，空载 Expert 不受抑制
show("skewed L_aux", f"{loss.item():.4f}")
show("grad p[0,0]", f"{probs.grad[0, 0].item():.5f} = αN·f_0/T = {alpha * N / T}")
show("grad p[0,7]", f"{probs.grad[0, 7].item():.5f} (idle expert)")

balanced L_aux               0.0200 = αK = 0.02
skewed L_aux                 0.0196
grad p[0,0]                  0.00125 = αN·f_0/T = 0.00125
grad p[0,7]                  0.00000 (idle expert)


Top-K 的离散选择本身不可微；对概率元素的梯度是 $\partial L/\partial p_{t,i}=\alpha N f_i/T$。这里人为固定 Top-K 索引以隔离梯度路径，索引并非由这组随机概率选出；不要把两个损失数值解读为均衡状态的全局最小值。空载专家在这个固定 dispatch mask 下梯度为零。

## 4. Expert Capacity 与一次统一 dispatch

容量 $C=\lceil TK/N\cdot CF\rceil$ 是每个专家跨所有 Top-K 槽位共享的总预算。dispatch 按 slot、expert、token 顺序接收，因而较早槽位与较早 token 有优先级；丢弃后不重新归一化剩余权重。

In [8]:
def expert_capacity(T: int, num_experts: int, top_k: int, capacity_factor: float) -> int:
    return math.ceil(T * top_k / num_experts * capacity_factor)

In [9]:
def dispatch(
    x: torch.Tensor,
    experts: nn.ModuleList,
    indices: torch.Tensor,
    weights: torch.Tensor,
    capacity: int,
) -> torch.Tensor:
    y = torch.zeros_like(x)
    remaining = [capacity] * len(experts)
    for slot in range(indices.size(1)):
        for expert, layer in enumerate(experts):
            tokens = (indices[:, slot] == expert).nonzero().squeeze(-1)[:remaining[expert]]
            if tokens.numel():
                y[tokens] += weights[tokens, slot, None] * layer(x[tokens])
                remaining[expert] -= tokens.numel()
    return y

In [10]:
class MoELayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int,
        num_experts: int,
        top_k: int,
        alpha: float = 0.01,
        capacity_factor: float = 1.0,
    ) -> None:
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.alpha = alpha
        self.capacity_factor = capacity_factor

        self.router = Router(d_model, num_experts, top_k)

        # 等参拆分（扩容拆分时为 d_ff）
        expert_hidden = d_ff // num_experts
        self.experts = nn.ModuleList([
            FFN(d_model, expert_hidden, nn.GELU) for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: [T, d_model]
        probs, topk_idx, routing_weights = self.router(x)  # [T, N], [T, K], [T, K]
        capacity = expert_capacity(x.size(0), self.num_experts, self.top_k, self.capacity_factor)
        y = dispatch(x, self.experts, topk_idx, routing_weights, capacity)
        aux_loss = load_balancing_loss(probs, topk_idx, self.alpha)
        return y, aux_loss

In [11]:
import torch


set_seed(0)
T, N, K = 16, 4, 2
layer = MoELayer(d_model=8, d_ff=32, num_experts=N, top_k=K, capacity_factor=1.0)
capacity = expert_capacity(T, N, K, 1.0)
show("capacity", f"ceil(K·T/N) = {capacity}")

with torch.no_grad():
    layer.router.weight.weight.zero_()
    layer.router.weight.weight[0] = 2.0
    layer.router.weight.weight[1] = 1.0

calls = {e: 0 for e in range(N)}
handles = [
    expert.register_forward_hook(lambda m, inp, out, e=e: calls.__setitem__(e, calls[e] + inp[0].size(0)))
    for e, expert in enumerate(layer.experts)
]
y, _ = layer(torch.ones(T, 8))
for h in handles:
    h.remove()

show("tokens per expert", calls)
show("fully dropped", f"{(y.abs().sum(-1) == 0).sum().item()} / {T}")

capacity                     ceil(K·T/N) = 8
tokens per expert            {0: 8, 1: 8, 2: 0, 3: 0}
fully dropped                8 / 16


输出展示每个专家总调用数不超过容量，以及完全溢出的 token。MoE 分支为零不等于整层输出必须为零：真实 Transformer 外部通常还有 residual。

## 5. Shared + Routed：教学版 DeepSeekMoE

共享专家处理全部 token，路由专家仍受 Top-K 与容量控制。这里把总宽度按 routed+shared 专家数拆分，单专家宽度为 `d_ff/(routed+shared)`。

In [12]:
class DeepSeekMoELayer(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int,
        num_routed_experts: int,
        num_shared_experts: int,
        top_k: int,
        alpha: float = 0.01,
        capacity_factor: float = 1.0,
    ) -> None:
        super().__init__()
        self.num_routed_experts = num_routed_experts
        self.top_k = top_k
        self.alpha = alpha
        self.capacity_factor = capacity_factor

        self.router = Router(d_model, num_routed_experts, top_k)

        # Shared Expert 与 Routed Expert 属于同一个专家池，等参拆分时计入总数
        expert_hidden = d_ff // (num_routed_experts + num_shared_experts)
        self.routed_experts = nn.ModuleList([
            FFN(d_model, expert_hidden, nn.GELU) for _ in range(num_routed_experts)
        ])
        self.shared_experts = nn.ModuleList([
            FFN(d_model, expert_hidden, nn.GELU) for _ in range(num_shared_experts)
        ])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: [T, d_model]
        probs, topk_idx, routing_weights = self.router(x)  # [T, N], [T, K], [T, K]
        capacity = expert_capacity(x.size(0), self.num_routed_experts, self.top_k, self.capacity_factor)
        routed_out = dispatch(x, self.routed_experts, topk_idx, routing_weights, capacity)
        shared_out = torch.zeros_like(x)
        for expert in self.shared_experts:
            shared_out += expert(x)
        aux_loss = load_balancing_loss(probs, topk_idx, self.alpha)
        return shared_out + routed_out, aux_loss

In [13]:
import torch


set_seed(0)
T, d_model, d_ff = 16, 8, 64
layer = DeepSeekMoELayer(d_model, d_ff, num_routed_experts=6, num_shared_experts=2, top_k=2)
show("expert hidden", f"{layer.routed_experts[0].mlp[0].out_features} = d_ff/(routed+shared)")

seen, handles = {}, []
for name, experts in [("shared", layer.shared_experts), ("routed", layer.routed_experts)]:
    for e, expert in enumerate(experts):
        handles.append(expert.register_forward_hook(
            lambda m, inp, out, n=f"{name}{e}": seen.__setitem__(n, seen.get(n, 0) + inp[0].size(0))
        ))
y, aux = layer(torch.randn(T, d_model))
for handle in handles:
    handle.remove()

for n in sorted(seen):
    show(f"{n} tokens", seen[n])
show("output / aux_loss", f"{tuple(y.shape)} / {aux.item():.4f}")

expert hidden                8 = d_ff/(routed+shared)
routed0 tokens               5
routed1 tokens               4
routed2 tokens               6
routed3 tokens               6
routed4 tokens               6
routed5 tokens               5
shared0 tokens               16
shared1 tokens               16
output / aux_loss            (16, 8) / 0.0201


hook 计数验证 shared 专家始终看见全部 token，并在使用后显式移除。这个小模型只讲机制，不是生产 DeepSeek 架构的完整复刻。